In [3]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('train.txt',sep=';',header=None, names=['text','emotion'])

In [7]:
df

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [8]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [9]:
unique_emotions = df['emotion'].unique()
emotion_num = {}
i=0
for emo in unique_emotions:
    emotion_num[emo] = i
    i+=1

In [10]:
df['emotion'] = df['emotion'].map(emotion_num)
emotion_num

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [11]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


**makinn the text lowercasing**

In [12]:
df['text'] = df['text'].str.lower()

In [13]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


**removing punctuations**


In [14]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

**remove numbers**

In [15]:
def remove_numbers(txt):
    return txt.translate(str.maketrans('', '', string.digits))

df['text'] = df['text'].apply(remove_numbers)

**remove emojis**

In [16]:
def remove_emojis(txt):
    #return txt.encode('ascii', 'ignore').decode('ascii')
    #or
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

**remove stopwords**

In [17]:
def remove_stopwords(txt):
    stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves',
                 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself',
                 'they', 'them', 'their', 'theirs', 'themselves',
                 'what', 'which', 'who', 'whom',
                 'this', 'that', "that'll", 
                 'these', 
                 'am','is','are','was','were','be','been','being',
                 'have','has','had','having',
                 'do','does','did','doing',
                 'a','an','the',
                 'and','but','if','or','because','as','until','while',
                 'of','at','by','for','with','about','against',
                 'between','into','through',
                 ]
    txt = [word for word in txt.split() if word not in stopwords]
    return " ".join(txt)



**remove stopwords 2nd method**

In [18]:
import nltk 
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


from nltk.tokenize import word_tokenize
nltk.download('punkt')

def remove_stopwords_nltk(txt):
    word_tokens = word_tokenize(txt)
    cleaned_txt = []
    for i in word_tokens:
        if i not in stop_words:
            cleaned_txt.append(i)    
    return " ".join(cleaned_txt)

df['text'] = df['text'].apply(remove_stopwords_nltk)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vijay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


**remove links and urls**

In [19]:
import re

def remove_links(txt):
    return re.sub(r'http\S+|www\S+', '', txt)

df['text'] = df['text'].apply(remove_links)

In [20]:
df


,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
15995,brief time beanbag said anna feel like beaten,0
15996,turning feel pathetic still waiting tables sub...,0
15997,feel strong good overall,5
15998,feel like rude comment im glad,1


In [21]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer

In [26]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

**NAIVE_BAYES MODEL for CountVectorizer**

In [30]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_bow, y_train)
y_pred = nb_classifier.predict(X_test_bow)

In [32]:
y_pred

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [ ]:
print(accuracy_score(y_test, y_pred))

0.7678125


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.26      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



**NAIVE_BAYES MODEL for TfidfVectorizer**

In [38]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train) 
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb_classifier_tfidf = MultinomialNB()
nb_classifier_tfidf.fit(X_train_tfidf, y_train) 
y_pred_tfidf = nb_classifier_tfidf.predict(X_test_tfidf)
y_pred_tfidf

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [42]:
print(accuracy_score(y_test, y_pred_tfidf))

0.6609375


In [43]:
print(classification_report(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200



**logistic regression MODEL for CountVectorizer**

In [50]:
from sklearn.linear_model import LogisticRegression


logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_bow, y_train)
y_pred_logistic_bow = logistic_model.predict(X_test_bow)
print(accuracy_score(y_test, y_pred_logistic_bow))

0.8884375


**logistic regression MODEL for TfidfVectorizer**

In [51]:
logistic_model.fit(X_train_tfidf, y_train)
y_pred_logistic_tfidf = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred_logistic_tfidf))

0.8621875


***this is sentiment analysis***

**emotion detection**